# 1. Train Scale Experiments

This notebook uses the local AttnResGPT-next-scale repo directly and runs the first-run, standard sweeps, or the new large 256/512 sweep without Google Drive.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

def find_repo_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path('/content/AttnResGPT-next-scale')]
    for candidate in candidates:
        if (candidate / 'src' / 'training' / 'train.py').exists() and (candidate / 'requirements.txt').exists():
            return candidate.resolve()
    raise FileNotFoundError(
        'Could not find AttnResGPT-next-scale. Open the notebook from the repo root or sync the repo to /content/AttnResGPT-next-scale.'
    )

REPO_DIR = find_repo_root()
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(f'Using repo at: {REPO_DIR}')


In [ ]:
import torch

print('cuda_available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device_name =', torch.cuda.get_device_name(0))
    print('bf16_supported =', torch.cuda.is_bf16_supported())


## Required First-Run Experiment

This runs SMALL only, contexts 128 and 512, baseline plus AttnRes, 300 steps each.


In [ ]:
!python experiments/scale_experiment.py --config configs/first_run.yaml --skip-existing


In [ ]:
import pandas as pd
from pathlib import Path

summary_df = pd.read_csv('outputs/logs/run_summaries.csv')
paired_df = pd.read_csv('outputs/logs/paired_comparisons.csv')
display(summary_df[['model', 'size', 'context', 'val_loss', 'perplexity', 'second_half_loss', 'mean_activation_norm_last_layer', 'mean_early_contribution', 'mean_late_contribution']].sort_values(['size', 'context', 'model']))
display(paired_df.sort_values(['size', 'context']))


## Optional Standard Sweeps

Use the helper below to launch additional local sweeps once the first-run outputs look healthy.


In [ ]:
import subprocess
import sys


def run_scale(config_path: str, *, skip_existing: bool = True) -> None:
    cmd = [sys.executable, 'experiments/scale_experiment.py', '--config', config_path]
    if skip_existing:
        cmd.append('--skip-existing')
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)


# Uncomment to launch the standard sweeps locally.
# run_scale('configs/small.yaml')
# run_scale('configs/medium.yaml')


## Large Sweep

This is the new large 256/512 experiment path for baseline and AttnRes. Outputs stay in the local `outputs/` directory, and the summary CSVs appear at `outputs/summary_large.csv` and `outputs/summary_large_comparison.csv`.


In [ ]:
# Uncomment to launch the large local sweep.
# run_scale('configs/large.yaml')

from pathlib import Path

if Path('outputs/summary_large.csv').exists():
    display(pd.read_csv('outputs/summary_large.csv').sort_values(['context', 'model']))
if Path('outputs/summary_large_comparison.csv').exists():
    display(pd.read_csv('outputs/summary_large_comparison.csv').sort_values(['context']))
